# Topic 3B: Logistic Regression for Credit Default
**Module 1 - Introduction to Machine Learning in Python**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, f1_score, classification_report)
import matplotlib.pyplot as plt


## 1. Prepare Credit Default Dataset
Using Kaggle 'Give Me Some Credit' dataset or synthetic data.


In [ ]:
# Generate synthetic credit data (replace with real data if available)
np.random.seed(42)
n = 20000
df = pd.DataFrame({
    'revolving_util': np.random.beta(2, 5, n) * 100,
    'age': np.random.normal(45, 12, n).clip(21, 85).astype(int),
    'times_30_59_late': np.random.poisson(0.3, n),
    'dti': np.random.lognormal(3.5, 0.5, n).clip(0, 100),
    'monthly_income': np.random.lognormal(8.5, 0.7, n).round(0),
    'open_credit_lines': np.random.poisson(8, n),
    'times_90_late': np.random.poisson(0.1, n),
    'num_real_estate': np.random.poisson(1, n),
    'times_60_89_late': np.random.poisson(0.15, n),
    'dependents': np.random.choice([0,1,2,3,4], n, p=[0.3,0.25,0.25,0.15,0.05]),
})

# Default probability based on features
logit = (-3 + 0.02*df['revolving_util'] - 0.03*df['age'] + 0.5*df['times_30_59_late']
         + 0.01*df['dti'] + 0.3*df['times_90_late'] + np.random.normal(0, 0.5, n))
from scipy.special import expit
df['default'] = (np.random.random(n) < expit(logit)).astype(int)
print(f'Default rate: {df["default"].mean():.4f}')


## 2. Train Logistic Regression


In [ ]:
features = [c for c in df.columns if c != 'default']
X = df[features]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_s, y_train)

y_proba = lr.predict_proba(X_test_s)[:, 1]
auroc = roc_auc_score(y_test, y_proba)
gini = 2 * auroc - 1
fpr, tpr, _ = roc_curve(y_test, y_proba)
ks = max(tpr - fpr)
auprc = average_precision_score(y_test, y_proba)

print(f'AUROC: {auroc:.4f}')
print(f'Gini:  {gini:.4f}')
print(f'KS:    {ks:.4f}')
print(f'AUPRC: {auprc:.4f}')


## 3. Coefficient Interpretation (Odds Ratios)


In [ ]:
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': lr.coef_[0],
    'Odds_Ratio': np.exp(lr.coef_[0]),
    'Abs_Coef': np.abs(lr.coef_[0])
}).sort_values('Abs_Coef', ascending=False)

print('Logistic Regression Coefficients and Odds Ratios:')
print(coef_df[['Feature', 'Coefficient', 'Odds_Ratio']].round(4).to_string(index=False))
print()
print('Interpretation examples:')
for _, row in coef_df.head(3).iterrows():
    direction = 'increases' if row['Coefficient'] > 0 else 'decreases'
    pct_change = abs(row['Odds_Ratio'] - 1) * 100
    print(f'  {row["Feature"]}: A 1-unit increase {direction} the odds of default by {pct_change:.1f}%')


## 4. ROC and Precision-Recall Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
ax1.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUROC = {auroc:.4f}')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()

# Precision-Recall Curve
prec, rec, thresholds_pr = precision_recall_curve(y_test, y_proba)
ax2.plot(rec, prec, color='indianred', linewidth=2, label=f'AUPRC = {auprc:.4f}')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()

plt.tight_layout()
plt.show()


## 5. Regularization Parameter Tuning


In [ ]:
C_values = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
cv_aurocs = []

for C in C_values:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42, class_weight='balanced')
    scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='roc_auc')
    cv_aurocs.append(scores.mean())
    print(f'C={C:>6}: CV AUROC = {scores.mean():.4f} (+/- {scores.std():.4f})')

best_C = C_values[np.argmax(cv_aurocs)]
print(f'\nBest C: {best_C}')

plt.figure(figsize=(8, 4))
plt.plot(C_values, cv_aurocs, 'o-', color='steelblue')
plt.xscale('log')
plt.xlabel('Regularization Parameter C')
plt.ylabel('CV AUROC')
plt.title('Regularization Tuning')
plt.axvline(x=best_C, color='red', linestyle='--', label=f'Best C={best_C}')
plt.legend()
plt.tight_layout()
plt.show()


## 6. Adverse Action Reasons (ECOA Compliance)


In [ ]:
# For a specific applicant, show which features contributed most to the prediction
applicant_idx = 0  # First test applicant
applicant = X_test.iloc[applicant_idx]
applicant_scaled = X_test_s[applicant_idx]
pred_proba = y_proba[applicant_idx]

# Feature contributions = coefficient * scaled feature value
contributions = lr.coef_[0] * applicant_scaled
contrib_df = pd.DataFrame({
    'Feature': features,
    'Value': applicant.values,
    'Contribution': contributions
}).sort_values('Contribution', ascending=False)

print(f'Applicant Predicted PD: {pred_proba:.4f}')
print(f'Actual Default: {y_test.iloc[applicant_idx]}')
print(f'\nTop factors INCREASING default risk (adverse action reasons):')
top_adverse = contrib_df[contrib_df['Contribution'] > 0].head(4)
for i, (_, row) in enumerate(top_adverse.iterrows(), 1):
    print(f'  {i}. {row["Feature"]} (value: {row["Value"]:.2f}, contribution: +{row["Contribution"]:.4f})')
